# 03 · MCP 함수 직접 호출로 데이터 가져오기

**파이프라인 3/4** — 에이전트/LLM 없이 **MCP 툴을 직접 호출**해 원시 데이터를 가져옵니다.
각 서버가 어떤 툴을 제공하고 무엇을 반환하는지 감을 잡는 단계입니다. (pipeline 패키지 없이 독립 실행)

## 셋업 (설정 + 인증 + MCP 헬퍼)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
TOKEN_CACHE = PROJECT_ROOT / '.token_cache.json'   # 로그인 1회 → 이후 노트북/웹앱이 재사용

# 2) 설정값 (.env에서 읽음)
TENANT_ID     = os.environ.get('TENANT_ID') or '00000000-0000-0000-0000-000000000000'
CLIENT_ID     = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AUTH_MODE     = os.environ.get('AUTH_MODE', 'device_code')   # device_code | client_credentials
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'

def _mcp_url(env_var, server_id):
    return (os.environ.get(env_var) or '').strip() or \
        f'https://agent365.svc.cloud.microsoft/agents/tenants/{TENANT_ID}/servers/{server_id}'

# 두 MCP 소스: Mail / Teams. 서버마다 delegated 스코프가 달라 토큰도 소스별로 1개씩.
SOURCES = {
    'mail':  {'label': 'Mail',  'url': _mcp_url('MAIL_MCP_SERVER_URL',  'mcp_MailTools')},
    'teams': {'label': 'Teams', 'url': _mcp_url('TEAMS_MCP_SERVER_URL', 'mcp_TeamsServer')},
}
for _s in SOURCES.values():
    _s['scopes'] = [f"{_s['url']}/.default"]

# 3) MSAL 인증 — 소스별 access token. device_code는 최초 1회만 로그인, 이후 캐시 재사용.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token(source_key: str) -> str:
    """소스 1개의 access token 확보 (없으면 device-code 로그인)."""
    assert CLIENT_ID, 'CLIENT_ID가 .env에 없습니다 (.env.example 참고).'
    scopes = SOURCES[source_key]['scopes']
    if AUTH_MODE == 'client_credentials':           # 앱 전용 토큰
        app = msal.ConfidentialClientApplication(
            CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET, token_cache=_cache)
        res = app.acquire_token_for_client(scopes=scopes)
    else:                                           # 위임(사용자) 로그인
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accounts = app.get_accounts()
        res = app.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not (res and 'access_token' in res):     # 캐시에 없으면 device-code 로그인
            flow = app.initiate_device_flow(scopes=scopes)
            print(flow['message'])                   # https://microsoft.com/devicelogin + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TENANT_ID   :', TENANT_ID, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
for _k, _s in SOURCES.items():
    print(f"  [{_k}] {_s['label']} -> {_s['url']}")

In [ ]:
# --- MCP 읽기 헬퍼: 연결은 호출할 때마다 열고 항상 닫는다 ---
async def _with_session(source_key, token, fn):
    url = SOURCES[source_key]['url']
    # 최신 MCP SDK: 인증 헤더는 httpx 클라이언트에 담아 streamable_http_client에 전달
    async with create_mcp_http_client(headers={'Authorization': f'Bearer {token}'}) as http_client:
        async with streamable_http_client(url, http_client=http_client) as (r, w, _):
            async with ClientSession(r, w) as s:      # MCP 세션 초기화(handshake)
                await s.initialize()
                return await fn(s)

async def list_tools(token, source_key):
    """MCP 서버가 노출하는 도구(함수) 목록."""
    res = await _with_session(source_key, token, lambda s: s.list_tools())
    return list(res.tools or [])

async def call_tool(token, source_key, name, args=None):
    """MCP 도구를 이름+인자로 직접 호출."""
    return await _with_session(source_key, token, lambda s: s.call_tool(name, arguments=args or {}))

def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def content_to_text(result) -> str:
    """MCP 도구 결과의 content 블록들을 읽기 좋은 문자열로 평탄화."""
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            parts.append(getattr(b, 'text', ''))
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

def readable(result):
    # 도구 결과를 사람이 보기 좋게: 텍스트 블록이 JSON(\uXXXX)이면 풀어서 한글로,
    # 자연어 요약(reply)이 있으면 그 부분만, 없으면 보기 좋은 JSON을 반환.
    replies, others = [], []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text' and getattr(b, 'text', ''):
            try:
                obj = json.loads(b.text)
            except Exception:
                others.append(b.text); continue
            if isinstance(obj, dict) and obj.get('reply'):
                replies.append(obj['reply'])
            else:
                others.append(json.dumps(obj, ensure_ascii=False, indent=2))
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            others.append(json.dumps(data, ensure_ascii=False, indent=2))
    text = '\n'.join(replies) if replies else '\n'.join(others)
    return ('ERROR: ' + text) if getattr(result, 'isError', False) else text

## 1. 토큰 + 툴 목록

In [ ]:
tokens = {}
for key in SOURCES:
    try:
        tokens[key] = ensure_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')

tools_by_source = {}
for key, token in tokens.items():
    tools_by_source[key] = await list_tools(token, key)
    print(f'{key}: ' + ', '.join(t.name for t in tools_by_source[key]))

## 2. 검색/조회 툴 후보 찾기
이름에 search/list/get/find/message/mail 등이 포함된 읽기 툴을 추려봅니다.

In [ ]:
import re
READ_HINT = re.compile(r'(search|list|get|find|read|message|mail|chat|channel)', re.I)
for key, ts in tools_by_source.items():
    print(f'\n=== {key} 읽기 툴 후보 ===')
    for t in ts:
        if READ_HINT.search(t.name):
            print(' -', t.name)

## 3. 파라미터 없이 간단히 목록 보기
가장 단순한 예시입니다. 입력값이 필요 없는 '목록' 툴은 `args={}` 로 바로 호출됩니다.
(메일은 목록 조회에도 검색 질의가 필요해, 이어지는 **4번**에서 다룹니다.)

In [ ]:
# 파라미터가 필요 없는 '목록' 툴을 그대로 호출 (args = {})
SIMPLE = [
    {'source': 'teams', 'tool': 'ListChats'},   # 내 Teams 채팅 목록 (Notes to Self 포함)
    {'source': 'teams', 'tool': 'ListTeams'},   # 내가 속한 팀 목록
]
for r in SIMPLE:
    print(f"\n===== [{r['source']}] {r['tool']} (args={{}}) =====")
    if not tokens.get(r['source']):
        print('  토큰 없음 — 위 1번 셀을 먼저 실행하세요.'); continue
    if not any(t.name == r['tool'] for t in tools_by_source.get(r['source'], [])):
        print('  이 서버 목록에 없음 — 2번 후보에서 이름 확인'); continue
    try:
        result = await call_tool(tokens[r['source']], r['source'], r['tool'], {})
        print(readable(result)[:2000] or '(빈 응답)')
    except Exception as exc:
        print('  실패:', exc)

## 4. 자연어 검색으로 2번 샘플 되읽기
`READS`에 **mail·teams 소스와 자연어 검색 툴, 질의어가 미리 채워져** 있습니다.
그대로 실행하면 `02_seed_sample_data`에서 주입한 메일/Teams 샘플을 되읽어옵니다.
(다른 데이터를 보려면 `message` 질의어만 바꾸면 됩니다.)

In [ ]:
# 2번에서 주입한 샘플을 되읽어올 (source, tool, args) 를 미리 채워둠.
# 두 소스 모두 '자연어 검색' 툴이라 message 에 찾을 내용을 문장으로 적으면 됩니다.
READS = [
    {'source': 'mail',
     'tool':   'SearchMessages',        # 메일 자연어 검색
     'args':   {'message': 'Postgres 커넥션 풀 튜닝과 k8s CrashLoopBackOff(파드) 관련 최근 이메일 찾아줘'}},
    {'source': 'teams',
     'tool':   'SearchTeamsMessages',   # Teams 자연어 검색 (Notes to Self 포함)
     'args':   {'message': 'GitHub Actions 캐시 히트율과 사내 로그 표준(JSON, trace_id) 관련 메시지 찾아줘'}},
]

# 미리 채운 툴이 실제 목록에 있는지 확인하고 입력 스키마를 출력 (인자 이름 참고용)
for r in READS:
    tool = next((t for t in tools_by_source.get(r['source'], []) if t.name == r['tool']), None)
    print(f"[{r['source']}] {r['tool']} {'✓' if tool else '✗ (2번 후보 목록에서 이름 확인 필요)'}")
    if tool:
        print('  스키마:', json.dumps(tool_schema(tool), ensure_ascii=False))

In [ ]:
# 미리 채운 READS 를 순서대로 호출 → 2번에서 넣은 데이터가 그대로 조회됩니다.
for r in READS:
    print(f"\n===== [{r['source']}] {r['tool']} =====")
    if not tokens.get(r['source']):
        print('  토큰 없음 — 위 1번 셀을 먼저 실행하세요.'); continue
    try:
        result = await call_tool(tokens[r['source']], r['source'], r['tool'], r['args'])
        print(readable(result)[:4000] or '(빈 응답)')
    except Exception as exc:
        print('  실패:', exc)

✅ 원하는 데이터를 직접 가져올 수 있음을 확인했습니다. 다음은 이 과정을 자연어로
자동화하는 **04_nl_aggregate_to_md** 입니다.